# 02 — Improved DDPM (IDDPM)

**Paper:** *Improved Denoising Diffusion Probabilistic Models* (Nichol & Dhariwal, 2021)  
**arXiv:** https://arxiv.org/abs/2102.09672

---

## What DDPM Left on the Table

The original DDPM (Ho et al., 2020) introduced the diffusion framework but used:
1. **Fixed linear noise schedule** — small values of `t` add very little noise but incur a large number of denoising steps
2. **Fixed posterior variance** `σ²_t = β_t` — not learned, limiting sample quality
3. **Simplified ELBO** — dropped a `L_T` constant term; good for sampling, but NLL is not tight

**IDDPM** fixes all three, achieving near-lossless compression rates competitive with autoregressive models while maintaining fast sampling.

## Improvement 1: Cosine Noise Schedule

DDPM uses a linear schedule: `β_t = linspace(β_1, β_T)`.  
The cosine schedule ensures `ᾱ_t` changes smoothly, avoiding regions where almost no noise is added.

![Cosine vs Linear Schedule](./figures/linear_cosine.png)

```
ᾱ_t = cos²( (t/T + s) / (1 + s) · π/2 )   where s = 0.008 (offset)
β_t = clip(1 - ᾱ_t / ᾱ_{t-1}, max=0.999)
```

The **offset** `s` prevents `β_t` from becoming too small near `t=0`, which would cause numerical issues.

![IDDPM Schedule Comparison](./figures/iddpm_schedule.png)

## Improvement 2: Learned Reverse Variance

DDPM fixes the reverse variance: `Σ_θ(x_t, t) = σ²_t · I`.  
IDDPM learns it by interpolating between the two extremes in **log space**:

```
Σ_θ = exp(v · log β_t + (1-v) · log β̃_t)
```

where `β̃_t = (1-ᾱ_{t-1})/(1-ᾱ_t) · β_t` is the lower bound.

`v ∈ [0,1]` is predicted by the model — one output per channel alongside the noise prediction ε.  
This lets the model learn exactly how much variance to use at each step.

## Improvement 3: Hybrid Loss Function

DDPM trains with simplified MSE on ε: `L_simple = ||ε - ε_θ||²`  
But this has poor negative log-likelihood (NLL) because it ignores the variance term.

IDDPM uses a **hybrid loss**:

```
L_hybrid = L_simple + λ · L_VLB
```

where `L_VLB` is the full variational lower bound and `λ = 0.001`.  
This trains `ε_θ` with the simple MSE (for stable training) while simultaneously training `v` (the variance output) with the proper probabilistic objective.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# ── Cosine Noise Schedule ──

def cosine_schedule(T=1000, s=0.008):
    t = torch.arange(T + 1, dtype=torch.float64)
    f = torch.cos((t / T + s) / (1 + s) * math.pi / 2) ** 2
    alphas_bar = f / f[0]
    betas = 1 - alphas_bar[1:] / alphas_bar[:-1]
    betas = betas.clamp(max=0.999)
    return betas.float()

def linear_schedule(T=1000, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)

T = 1000
betas_lin = linear_schedule(T)
betas_cos = cosine_schedule(T)

alphas_bar_lin = (1 - betas_lin).cumprod(0)
alphas_bar_cos = (1 - betas_cos).cumprod(0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_range = torch.arange(T)
axes[0].plot(t_range, betas_lin.numpy(), label='Linear β_t', color='steelblue')
axes[0].plot(t_range, betas_cos.numpy(), label='Cosine β_t', color='tomato')
axes[0].set_title('Noise Schedule: β_t'); axes[0].legend(); axes[0].set_xlabel('Timestep t')

axes[1].plot(t_range, alphas_bar_lin.numpy(), label='Linear ᾱ_t', color='steelblue')
axes[1].plot(t_range, alphas_bar_cos.numpy(), label='Cosine ᾱ_t', color='tomato')
axes[1].set_title('Cumulative: ᾱ_t'); axes[1].legend(); axes[1].set_xlabel('Timestep t')

plt.suptitle('DDPM (Linear) vs IDDPM (Cosine) Noise Schedule', fontsize=13)
plt.tight_layout(); plt.show()

print(f"Linear: at t=500, ᾱ = {alphas_bar_lin[500]:.4f}")
print(f"Cosine: at t=500, ᾱ = {alphas_bar_cos[500]:.4f}")

In [ ]:
# ── IDDPM Diffusion with Learned Variance ──

class IDDPMSchedule:
    def __init__(self, T=1000, s=0.008):
        self.T = T
        betas = cosine_schedule(T, s)
        alphas = 1.0 - betas
        alphas_bar = alphas.cumprod(0)
        alphas_bar_prev = F.pad(alphas_bar[:-1], (1, 0), value=1.0)
        posterior_var = betas * (1 - alphas_bar_prev) / (1 - alphas_bar)

        self.betas           = betas
        self.alphas          = alphas
        self.alphas_bar      = alphas_bar
        self.alphas_bar_prev = alphas_bar_prev
        self.sqrt_abar       = alphas_bar.sqrt()
        self.sqrt_1m_abar    = (1 - alphas_bar).sqrt()
        self.posterior_var   = posterior_var.clamp(min=1e-20)
        self.log_beta        = betas.log()
        self.log_post_var    = self.posterior_var.log()

    def q_sample(self, x0, t, noise=None):
        if noise is None: noise = torch.randn_like(x0)
        sa  = self.sqrt_abar[t].view(-1,1,1,1)
        s1a = self.sqrt_1m_abar[t].view(-1,1,1,1)
        return sa * x0 + s1a * noise, noise

    def learned_variance(self, v, t):
        """
        v: (B, C, H, W) in [0,1] (model output after sigmoid)
        Interpolate between log β_t and log β̃_t in log space.
        """
        log_beta  = self.log_beta[t].view(-1,1,1,1)
        log_bpost = self.log_post_var[t].view(-1,1,1,1)
        return torch.exp(v * log_beta + (1 - v) * log_bpost)

    def vlb_loss(self, x0, xt, t, eps_pred, v_pred):
        """
        Variational Lower Bound loss for the learned variance term.
        Uses KL between q(x_{t-1}|x_t,x0) and p_theta(x_{t-1}|x_t).
        """
        # Posterior mean (from x0 and xt)
        abar   = self.alphas_bar[t].view(-1,1,1,1)
        abar_p = self.alphas_bar_prev[t].view(-1,1,1,1)
        beta   = self.betas[t].view(-1,1,1,1)
        pv     = self.posterior_var[t].view(-1,1,1,1)

        mu_q = (abar_p.sqrt() * beta / (1 - abar)) * x0 +                (self.alphas[t].view(-1,1,1,1).sqrt() * (1 - abar_p) / (1 - abar)) * xt

        # Model predicted x0
        x0_pred = (xt - self.sqrt_1m_abar[t].view(-1,1,1,1) * eps_pred) / abar.sqrt()
        x0_pred = x0_pred.clamp(-1, 1)
        mu_p = (abar_p.sqrt() * beta / (1 - abar)) * x0_pred +                (self.alphas[t].view(-1,1,1,1).sqrt() * (1 - abar_p) / (1 - abar)) * xt

        var_p = self.learned_variance(v_pred.sigmoid(), t)

        # Gaussian KL
        kl = 0.5 * ((mu_q - mu_p) ** 2 / var_p + var_p.log() - pv.log() - 1 + pv / var_p)
        return kl.mean()


schedule = IDDPMSchedule(T=1000)
print("Posterior var at t=0:  ", schedule.posterior_var[0].item())
print("Posterior var at t=999:", schedule.posterior_var[999].item())
print("Beta at t=999:         ", schedule.betas[999].item())

In [ ]:
# ── Simple IDDPM U-Net stub (shows the dual output: ε + v) ──

class IDDPMHead(nn.Module):
    """
    Final output head that predicts both noise (ε) and raw variance (v).
    Real IDDPM uses a full U-Net; here we just show the head design.
    
    out_channels = 2 * in_channels:
        first half  → ε (noise prediction)
        second half → v (raw variance logit, passed through sigmoid)
    """
    def __init__(self, in_channels=4, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 3, padding=1),
            nn.GroupNorm(8, hidden), nn.SiLU(),
            nn.Conv2d(hidden, in_channels * 2, 1)   # predict ε and v
        )
    
    def forward(self, x):
        out = self.net(x)
        eps_pred, v_pred = out.chunk(2, dim=1)
        return eps_pred, v_pred


# Hybrid loss
def iddpm_loss(model, schedule, x0, y=None, lambda_vlb=0.001):
    B = x0.shape[0]
    t = torch.randint(0, schedule.T, (B,))
    xt, eps = schedule.q_sample(x0, t)
    
    eps_pred, v_pred = model(xt)
    
    L_simple = F.mse_loss(eps_pred, eps)
    L_vlb    = schedule.vlb_loss(x0, xt, t, eps_pred.detach(), v_pred)
    return L_simple + lambda_vlb * L_vlb, L_simple.item(), L_vlb.item()


# Demo
head = IDDPMHead(in_channels=4)
x0 = torch.randn(2, 4, 16, 16)
loss, l_s, l_v = iddpm_loss(head, schedule, x0)
print(f"Total loss: {loss.item():.4f}   L_simple: {l_s:.4f}   L_VLB: {l_v:.4f}")

## Summary

| DDPM | IDDPM |
|------|-------|
| Linear β schedule | Cosine β schedule (smoother ᾱ_t) |
| Fixed posterior variance σ²_t | Learned variance (interpolate β_t, β̃_t) |
| Simple MSE loss only | Hybrid: `L_simple + 0.001 · L_VLB` |
| ~1000 steps needed | Enables good quality at 250 steps |
| NLL ≈ 3.7 bits/dim (CIFAR) | NLL ≈ **2.94** bits/dim (competitive with autoregressive) |

### Key Equations

**Cosine schedule:**
$$\bar\alpha_t = \frac{f(t)}{f(0)}, \quad f(t) = \cos^2\!\left(\frac{t/T + s}{1+s} \cdot \frac{\pi}{2}\right)$$

**Learned variance (log-space interpolation):**
$$\Sigma_\theta = \exp\!\left(v \log\beta_t + (1-v)\log\tilde\beta_t\right)$$

**Hybrid loss:**
$$L_{\text{hybrid}} = L_{\text{simple}} + \lambda \cdot L_{\text{VLB}}, \quad \lambda = 0.001$$